# LiDAR Obstacle Detection - PointNet Training (Colab Pro Optimized)
**Airbus AI Hackathon 2026**

Version optimisée pour Colab Pro avec GPU A100/V100.

## Optimisations
- Mixed Precision Training (AMP) - 2x plus rapide
- Batch size adaptatif selon le GPU
- Cache des données en RAM
- Gradient clipping pour stabilité
- OneCycleLR scheduler

## Instructions
1. **Runtime > Change runtime type > GPU (A100 ou V100)**
2. Exécutez les cellules dans l'ordre
3. Uploadez vos fichiers HDF5 dans Google Drive

## 1. Vérification GPU et Configuration Auto

In [ ]:
# Vérifier le GPU et configurer automatiquement
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    
    # Configuration automatique selon le GPU
    if 'A100' in gpu_name:
        GPU_CONFIG = {'batch_size': 16, 'n_points': 65536, 'num_workers': 4}
        print("\n>>> Configuration A100 détectée <<<")
    elif 'V100' in gpu_name:
        GPU_CONFIG = {'batch_size': 12, 'n_points': 49152, 'num_workers': 4}
        print("\n>>> Configuration V100 détectée <<<")
    elif 'T4' in gpu_name:
        GPU_CONFIG = {'batch_size': 8, 'n_points': 32768, 'num_workers': 2}
        print("\n>>> Configuration T4 détectée <<<")
    else:
        GPU_CONFIG = {'batch_size': 8, 'n_points': 32768, 'num_workers': 2}
        print("\n>>> Configuration par défaut <<<")
    
    print(f"Batch size: {GPU_CONFIG['batch_size']}")
    print(f"Points per frame: {GPU_CONFIG['n_points']}")
else:
    raise RuntimeError("GPU non disponible! Activez le GPU dans Runtime > Change runtime type")

In [ ]:
# Installer les dépendances
!pip install h5py pandas tqdm -q

## 2. Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Créer les dossiers nécessaires
!mkdir -p /content/drive/MyDrive/LiDAR_Hackathon/data
!mkdir -p /content/drive/MyDrive/LiDAR_Hackathon/checkpoints

DATA_DIR = "/content/drive/MyDrive/LiDAR_Hackathon/data"
CHECKPOINT_DIR = "/content/drive/MyDrive/LiDAR_Hackathon/checkpoints"

# Vérifier les fichiers
h5_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.h5')]
print(f"\nFichiers HDF5 trouvés: {len(h5_files)}")
for f in sorted(h5_files):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024*1024)
    print(f"  - {f} ({size_mb:.1f} MB)")

if len(h5_files) == 0:
    print("\n" + "="*50)
    print("IMPORTANT: Uploadez vos fichiers scene_*.h5 dans:")
    print(DATA_DIR)
    print("="*50)

## 3. Code Source (Optimisé)

In [ ]:
# ============================================
# lidar_utils.py
# ============================================

import h5py
import numpy as np
import pandas as pd

def load_h5_data(file_path, dataset_name="lidar_points"):
    """Loads HDF5 data into a Pandas DataFrame."""
    with h5py.File(file_path, "r") as f:
        if dataset_name not in f:
            raise ValueError(f"Dataset '{dataset_name}' not found in {file_path}")
        points = f[dataset_name][:]
    return pd.DataFrame({name: points[name] for name in points.dtype.names})

def get_unique_poses(df):
    """Returns a DataFrame of unique poses with a 'pose_index'."""
    pose_fields = ["ego_x", "ego_y", "ego_z", "ego_yaw"]
    if not all(f in df.columns for f in pose_fields):
        return None
    pose_counts = (
        df.groupby(pose_fields)
        .size()
        .reset_index(name="num_points")
        .reset_index(names="pose_index")
    )
    return pose_counts

def filter_by_pose(df, pose_row):
    """Filters the dataframe for a specific pose quadruplet."""
    return df[
        (df["ego_x"] == pose_row["ego_x"]) &
        (df["ego_y"] == pose_row["ego_y"]) &
        (df["ego_z"] == pose_row["ego_z"]) &
        (df["ego_yaw"] == pose_row["ego_yaw"])
    ].reset_index(drop=True)

def spherical_to_local_cartesian(df):
    """Converts spherical coordinates to local Cartesian (Lidar Frame)."""
    distance_m = df["distance_cm"].to_numpy() / 100.0
    azimuth_rad = np.radians(df["azimuth_raw"] / 100.0)
    elevation_rad = np.radians(df["elevation_raw"] / 100.0)

    x = distance_m * np.cos(elevation_rad) * np.cos(azimuth_rad)
    y = -distance_m * np.cos(elevation_rad) * np.sin(azimuth_rad)
    z = distance_m * np.sin(elevation_rad)

    return np.column_stack((x, y, z))

print("lidar_utils loaded")

In [ ]:
# ============================================
# dataset.py (Optimisé avec cache)
# ============================================

from torch.utils.data import Dataset
from collections import OrderedDict

# Class color mapping (RGB values)
CLASS_COLORS = {
    0: (38, 23, 180),      # Antenna
    1: (177, 132, 47),     # Cable
    2: (129, 81, 97),      # Electric pole
    3: (66, 132, 9),       # Wind turbine
}

CLASS_NAMES = {
    0: "Antenna",
    1: "Cable",
    2: "Electric pole",
    3: "Wind turbine",
    4: "Background"
}

NUM_CLASSES = 5  # 4 obstacles + 1 background


def rgb_array_to_class_ids(rgb_array):
    """Convert array of RGB values to class IDs (vectorized)."""
    r, g, b = rgb_array[:, 0], rgb_array[:, 1], rgb_array[:, 2]
    class_ids = np.full(len(rgb_array), 4, dtype=np.int64)  # Default: background

    for class_id, (cr, cg, cb) in CLASS_COLORS.items():
        mask = (r == cr) & (g == cg) & (b == cb)
        class_ids[mask] = class_id

    return class_ids


class LRUCache:
    """Simple LRU cache for HDF5 files."""
    def __init__(self, maxsize=5):
        self.cache = OrderedDict()
        self.maxsize = maxsize
    
    def get(self, key):
        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]
        return None
    
    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        else:
            if len(self.cache) >= self.maxsize:
                self.cache.popitem(last=False)
            self.cache[key] = value


class LidarDatasetCached(Dataset):
    """PyTorch Dataset with LRU cache for faster loading."""

    def __init__(self, data_dir, n_points=32768, augment=True, normalize=True, 
                 use_reflectivity=True, cache_size=5):
        self.data_dir = data_dir
        self.n_points = n_points
        self.augment = augment
        self.normalize = normalize
        self.use_reflectivity = use_reflectivity
        self.frames = []
        self.file_cache = LRUCache(maxsize=cache_size)
        self._load_all_frames()

    def _load_all_frames(self):
        """Load metadata for all frames from all HDF5 files."""
        h5_files = [f for f in os.listdir(self.data_dir) if f.endswith('.h5')]

        for h5_file in sorted(h5_files):
            file_path = os.path.join(self.data_dir, h5_file)
            try:
                df = load_h5_data(file_path)
                pose_counts = get_unique_poses(df)

                if pose_counts is not None:
                    for idx in range(len(pose_counts)):
                        self.frames.append({
                            'file_path': file_path,
                            'pose_index': idx,
                            'pose_data': pose_counts.iloc[idx].to_dict()
                        })
            except Exception as e:
                print(f"Warning: Could not load {h5_file}: {e}")

        print(f"Loaded {len(self.frames)} frames from {len(h5_files)} files")

    def __len__(self):
        return len(self.frames)

    def _get_file_data(self, file_path):
        """Get file data from cache or load it."""
        cached = self.file_cache.get(file_path)
        if cached is not None:
            return cached
        
        df = load_h5_data(file_path)
        pose_counts = get_unique_poses(df)
        self.file_cache.put(file_path, (df, pose_counts))
        return df, pose_counts

    def _augment_points(self, xyz):
        """Apply data augmentation to point cloud."""
        # Random rotation around Z-axis
        theta = np.random.uniform(0, 2 * np.pi)
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        rotation_matrix = np.array([
            [cos_t, -sin_t, 0],
            [sin_t, cos_t, 0],
            [0, 0, 1]
        ], dtype=np.float32)
        xyz = xyz @ rotation_matrix.T
        
        # Random jitter
        xyz = xyz + np.random.normal(0, 0.02, size=xyz.shape).astype(np.float32)
        
        # Random scale
        scale = np.random.uniform(0.9, 1.1)
        xyz = xyz * scale
        
        # Random flip (X or Y axis)
        if np.random.random() > 0.5:
            xyz[:, 0] = -xyz[:, 0]
        if np.random.random() > 0.5:
            xyz[:, 1] = -xyz[:, 1]
            
        return xyz

    def _normalize_points(self, xyz):
        """Normalize point cloud to unit sphere."""
        centroid = np.mean(xyz, axis=0)
        xyz = xyz - centroid
        max_dist = np.max(np.sqrt(np.sum(xyz ** 2, axis=1)))
        if max_dist > 0:
            xyz = xyz / max_dist
        return xyz

    def _sample_points(self, xyz, features, labels):
        """Sample or pad point cloud to fixed size."""
        n_points_current = xyz.shape[0]

        if n_points_current == 0:
            xyz = np.zeros((self.n_points, 3), dtype=np.float32)
            features = np.zeros(self.n_points, dtype=np.float32) if features is not None else None
            labels = np.full(self.n_points, 4, dtype=np.int64)
            return xyz, features, labels

        if n_points_current >= self.n_points:
            indices = np.random.choice(n_points_current, self.n_points, replace=False)
        else:
            indices = np.random.choice(n_points_current, self.n_points, replace=True)

        return xyz[indices], features[indices] if features is not None else None, labels[indices]

    def __getitem__(self, idx):
        frame_info = self.frames[idx]

        # Use cached file data
        df, pose_counts = self._get_file_data(frame_info['file_path'])
        selected_pose = pose_counts.iloc[frame_info['pose_index']]
        frame_df = filter_by_pose(df, selected_pose)

        # Filter valid points
        valid_mask = frame_df['distance_cm'] > 0
        frame_df = frame_df[valid_mask].reset_index(drop=True)

        # Convert coordinates
        xyz = spherical_to_local_cartesian(frame_df).astype(np.float32)

        # Get labels
        rgb_values = frame_df[['r', 'g', 'b']].values
        labels = rgb_array_to_class_ids(rgb_values)

        # Get reflectivity
        reflectivity = None
        if self.use_reflectivity and 'reflectivity' in frame_df.columns:
            reflectivity = frame_df['reflectivity'].values.astype(np.float32) / 255.0

        # Sample to fixed size
        xyz, reflectivity, labels = self._sample_points(xyz, reflectivity, labels)

        # Normalize
        if self.normalize:
            xyz = self._normalize_points(xyz)

        # Augment (training only)
        if self.augment:
            xyz = self._augment_points(xyz)

        # Combine features
        if self.use_reflectivity and reflectivity is not None:
            features = np.column_stack([xyz, reflectivity])
        else:
            features = xyz

        return torch.from_numpy(features).float(), torch.from_numpy(labels).long()

    def get_class_weights(self):
        """Compute class weights based on inverse frequency."""
        class_counts = np.zeros(NUM_CLASSES, dtype=np.float64)

        sample_size = min(30, len(self.frames))
        sample_indices = np.random.choice(len(self.frames), sample_size, replace=False)

        for idx in sample_indices:
            _, labels = self[idx]
            for c in range(NUM_CLASSES):
                class_counts[c] += (labels == c).sum().item()

        total = class_counts.sum()
        weights = total / (NUM_CLASSES * class_counts + 1e-6)
        weights = weights / weights.sum() * NUM_CLASSES

        return torch.from_numpy(weights).float()


def get_dataloaders(data_dir, batch_size=8, n_points=32768, num_workers=2, val_split=0.2):
    """Create train and validation dataloaders."""
    # Training dataset (with augmentation)
    train_dataset = LidarDatasetCached(
        data_dir=data_dir,
        n_points=n_points,
        augment=True,
        normalize=True,
        use_reflectivity=True,
        cache_size=10
    )
    
    # Split indices
    n_total = len(train_dataset)
    n_val = int(n_total * val_split)
    n_train = n_total - n_val
    
    # Random split
    indices = np.random.permutation(n_total)
    train_indices = indices[:n_train]
    val_indices = indices[n_train:]
    
    train_sampler = torch.utils.data.SubsetRandomSampler(train_indices)
    val_sampler = torch.utils.data.SubsetRandomSampler(val_indices)

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True if num_workers > 0 else False
    )

    val_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=val_sampler,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True if num_workers > 0 else False
    )

    return train_loader, val_loader, train_dataset.get_class_weights()

print("dataset loaded (with cache)")

In [ ]:
# ============================================
# pointnet_model.py
# ============================================

import torch.nn as nn
import torch.nn.functional as F


class TNet(nn.Module):
    """Transformation Network for learning spatial/feature transformations."""

    def __init__(self, k=3):
        super().__init__()
        self.k = k

        self.conv1 = nn.Conv1d(k, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 1024, 1)

        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, k * k)

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(1024)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(256)

    def forward(self, x):
        batch_size = x.size(0)

        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))

        x = torch.max(x, 2)[0]

        x = F.relu(self.bn4(self.fc1(x)))
        x = F.relu(self.bn5(self.fc2(x)))
        x = self.fc3(x)

        identity = torch.eye(self.k, device=x.device, dtype=x.dtype)
        identity = identity.view(1, self.k * self.k).repeat(batch_size, 1)

        x = x + identity
        x = x.view(-1, self.k, self.k)

        return x


class PointNetEncoder(nn.Module):
    """PointNet encoder with T-Net transformations."""

    def __init__(self, input_channels=4, feature_transform=True):
        super().__init__()
        self.feature_transform = feature_transform

        self.tnet3 = TNet(k=3)

        self.conv1 = nn.Conv1d(input_channels, 64, 1)
        self.conv2 = nn.Conv1d(64, 64, 1)

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(64)

        if feature_transform:
            self.tnet64 = TNet(k=64)

        self.conv3 = nn.Conv1d(64, 64, 1)
        self.conv4 = nn.Conv1d(64, 128, 1)
        self.conv5 = nn.Conv1d(128, 1024, 1)

        self.bn3 = nn.BatchNorm1d(64)
        self.bn4 = nn.BatchNorm1d(128)
        self.bn5 = nn.BatchNorm1d(1024)

    def forward(self, x):
        batch_size, n_points, _ = x.size()

        xyz = x[:, :, :3]

        xyz_t = xyz.transpose(2, 1)
        trans3 = self.tnet3(xyz_t)
        xyz = torch.bmm(xyz, trans3)

        if x.size(2) > 3:
            x = torch.cat([xyz, x[:, :, 3:]], dim=2)
        else:
            x = xyz

        x = x.transpose(2, 1)

        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))

        trans64 = None
        if self.feature_transform:
            trans64 = self.tnet64(x)
            x = x.transpose(2, 1)
            x = torch.bmm(x, trans64)
            x = x.transpose(2, 1)

        point_features = x

        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))

        global_feature = torch.max(x, 2)[0]

        return global_feature, point_features, trans3, trans64


class PointNetSegmentation(nn.Module):
    """PointNet for semantic segmentation."""

    def __init__(self, num_classes=5, input_channels=4, feature_transform=True):
        super().__init__()
        self.num_classes = num_classes
        self.feature_transform = feature_transform

        self.encoder = PointNetEncoder(
            input_channels=input_channels,
            feature_transform=feature_transform
        )

        self.conv1 = nn.Conv1d(1088, 512, 1)
        self.conv2 = nn.Conv1d(512, 256, 1)
        self.conv3 = nn.Conv1d(256, 128, 1)
        self.conv4 = nn.Conv1d(128, num_classes, 1)

        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.bn3 = nn.BatchNorm1d(128)

        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        batch_size, n_points, _ = x.size()

        global_feature, point_features, trans3, trans64 = self.encoder(x)

        global_feature = global_feature.unsqueeze(2).repeat(1, 1, n_points)
        concat_features = torch.cat([point_features, global_feature], dim=1)

        x = F.relu(self.bn1(self.conv1(concat_features)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.dropout(x)
        x = self.conv4(x)

        x = x.transpose(2, 1)

        return x, trans3, trans64


def feature_transform_regularizer(trans):
    """Regularization loss for feature transformation matrix."""
    if trans is None:
        return 0

    d = trans.size(1)
    batch_size = trans.size(0)

    identity = torch.eye(d, device=trans.device, dtype=trans.dtype)
    identity = identity.unsqueeze(0).repeat(batch_size, 1, 1)

    loss = torch.mean(torch.norm(torch.bmm(trans, trans.transpose(2, 1)) - identity, dim=(1, 2)))
    return loss


def get_model_size(model):
    """Return the number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("pointnet_model loaded")

## 4. Entraînement (Optimisé avec AMP)

In [ ]:
# ============================================
# Fonctions d'entraînement avec Mixed Precision
# ============================================

import time
from tqdm import tqdm
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler, autocast


def compute_iou(pred, target, num_classes):
    """Compute IoU per class."""
    ious = []
    pred = pred.reshape(-1)
    target = target.reshape(-1)

    for cls in range(num_classes):
        pred_cls = pred == cls
        target_cls = target == cls

        intersection = (pred_cls & target_cls).sum().float()
        union = (pred_cls | target_cls).sum().float()

        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append((intersection / union).item())

    return ious


def train_one_epoch_amp(model, train_loader, criterion, optimizer, scheduler, 
                        scaler, device, feature_reg_weight=0.001, grad_clip=1.0):
    """Train for one epoch with mixed precision."""
    model.train()
    total_loss = 0
    total_correct = 0
    total_points = 0

    pbar = tqdm(train_loader, desc="Training")
    for features, labels in pbar:
        features = features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)  # More efficient than zero_grad()

        # Mixed precision forward pass
        with autocast():
            outputs, trans3, trans64 = model(features)
            outputs_flat = outputs.reshape(-1, NUM_CLASSES)
            labels_flat = labels.reshape(-1)

            loss = criterion(outputs_flat, labels_flat)
            reg_loss = feature_transform_regularizer(trans64)
            loss = loss + feature_reg_weight * reg_loss

        # Scaled backward pass
        scaler.scale(loss).backward()
        
        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        
        scaler.step(optimizer)
        scaler.update()
        
        # Update OneCycleLR per batch
        scheduler.step()

        total_loss += loss.item()
        pred = outputs_flat.argmax(dim=1)
        total_correct += (pred == labels_flat).sum().item()
        total_points += labels_flat.numel()

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100 * total_correct / total_points:.2f}%',
            'lr': f'{scheduler.get_last_lr()[0]:.6f}'
        })

    return total_loss / len(train_loader), total_correct / total_points


def validate_amp(model, val_loader, criterion, device):
    """Validate the model with mixed precision."""
    model.eval()
    total_loss = 0
    total_correct = 0
    total_points = 0
    all_ious = []

    with torch.no_grad():
        for features, labels in tqdm(val_loader, desc="Validation"):
            features = features.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast():
                outputs, _, _ = model(features)
                outputs_flat = outputs.reshape(-1, NUM_CLASSES)
                labels_flat = labels.reshape(-1)
                loss = criterion(outputs_flat, labels_flat)

            total_loss += loss.item()
            pred = outputs_flat.argmax(dim=1)
            total_correct += (pred == labels_flat).sum().item()
            total_points += labels_flat.numel()

            pred_batch = outputs.argmax(dim=2)
            batch_ious = compute_iou(pred_batch, labels, NUM_CLASSES)
            all_ious.append(batch_ious)

    avg_loss = total_loss / len(val_loader)
    accuracy = total_correct / total_points

    all_ious = np.array(all_ious)
    mean_ious = np.nanmean(all_ious, axis=0)
    miou = np.nanmean(mean_ious[:4])  # mIoU for obstacle classes only

    return avg_loss, accuracy, mean_ious, miou

print("training functions loaded (with AMP)")

In [ ]:
# ============================================
# Configuration et lancement
# ============================================

# Hyperparamètres (auto-configurés selon GPU)
EPOCHS = 100
BATCH_SIZE = GPU_CONFIG['batch_size']
N_POINTS = GPU_CONFIG['n_points']
NUM_WORKERS = GPU_CONFIG['num_workers']
LEARNING_RATE = 0.001
MAX_LR = 0.01  # Pour OneCycleLR

device = torch.device("cuda")
print(f"Configuration:")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Points per frame: {N_POINTS}")
print(f"  - Workers: {NUM_WORKERS}")
print(f"  - Epochs: {EPOCHS}")

# Charger les données
print("\nLoading data...")
train_loader, val_loader, class_weights = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    n_points=N_POINTS,
    num_workers=NUM_WORKERS
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Class weights: {class_weights}")

# Créer le modèle
model = PointNetSegmentation(num_classes=NUM_CLASSES, input_channels=4)
model = model.to(device)
print(f"\nModel parameters: {get_model_size(model):,}")

# Loss et optimizer
class_weights = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

# OneCycleLR scheduler (meilleur que CosineAnnealing)
steps_per_epoch = len(train_loader)
scheduler = OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    pct_start=0.3,
    anneal_strategy='cos'
)

# Mixed precision scaler
scaler = GradScaler()

print("\nReady to train!")

In [ ]:
# ============================================
# Boucle d'entraînement
# ============================================

best_miou = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'miou': []}

print("\n" + "="*60)
print("Starting training with Mixed Precision (AMP)...")
print("="*60)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    start_time = time.time()

    # Train
    train_loss, train_acc = train_one_epoch_amp(
        model, train_loader, criterion, optimizer, scheduler, scaler, device
    )

    # Validate
    val_loss, val_acc, class_ious, miou = validate_amp(
        model, val_loader, criterion, device
    )

    epoch_time = time.time() - start_time

    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['miou'].append(miou)

    # Print metrics
    print(f"Time: {epoch_time:.1f}s")
    print(f"Train - Loss: {train_loss:.4f}, Acc: {100*train_acc:.2f}%")
    print(f"Val   - Loss: {val_loss:.4f}, Acc: {100*val_acc:.2f}%, mIoU: {100*miou:.2f}%")
    print("IoU per class:")
    for c in range(NUM_CLASSES):
        iou = class_ious[c]
        if not np.isnan(iou):
            print(f"  {CLASS_NAMES[c]}: {100*iou:.2f}%")

    # Save checkpoint
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'miou': miou,
        'best_miou': best_miou,
        'config': GPU_CONFIG
    }

    # Save latest
    torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, 'latest.pth'))

    # Save best
    if miou > best_miou:
        best_miou = miou
        checkpoint['best_miou'] = best_miou
        torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, 'best.pth'))
        print(f">>> New best mIoU: {100*best_miou:.2f}% <<<")

    # Save periodic checkpoint
    if (epoch + 1) % 20 == 0:
        torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, f'epoch_{epoch+1}.pth'))

print(f"\n" + "="*60)
print(f"Training complete! Best mIoU: {100*best_miou:.2f}%")
print(f"Checkpoints saved to: {CHECKPOINT_DIR}")
print("="*60)

In [ ]:
# ============================================
# Visualiser les courbes d'entraînement
# ============================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot([x*100 for x in history['val_acc']])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Validation Accuracy')
axes[1].grid(True)

# mIoU
axes[2].plot([x*100 for x in history['miou']])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('mIoU (%)')
axes[2].set_title(f'Validation mIoU (Best: {100*best_miou:.2f}%)')
axes[2].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 5. Export et Téléchargement

In [ ]:
# Télécharger le meilleur modèle
from google.colab import files

best_model_path = os.path.join(CHECKPOINT_DIR, 'best.pth')
if os.path.exists(best_model_path):
    files.download(best_model_path)
    print(f"Downloaded: {best_model_path}")
else:
    print("No best.pth found.")

In [ ]:
# Exporter en ONNX
import torch.onnx

# Charger le meilleur modèle
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best.pth'))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Input factice
dummy_input = torch.randn(1, N_POINTS, 4).to(device)

# Exporter
onnx_path = os.path.join(CHECKPOINT_DIR, 'pointnet_segmentation.onnx')
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['point_cloud'],
    output_names=['segmentation', 'trans3', 'trans64'],
    dynamic_axes={
        'point_cloud': {0: 'batch_size', 1: 'num_points'},
        'segmentation': {0: 'batch_size', 1: 'num_points'}
    },
    opset_version=11
)

print(f"ONNX saved: {onnx_path}")
files.download(onnx_path)